<a href="https://colab.research.google.com/github/Jigyasa-2606/Feature-Extraction-and-Classification-of-Moving-Vehicles-on-Highways/blob/main/AODnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
import numpy as np
import cv2
import os
import sys
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.utils import save_image
import torch.nn as nn
from torch.utils.data import DataLoader


In [58]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [59]:
torch.cuda.empty_cache()

In [60]:
import cv2
import os

video_path = "/content/Highway_5_fog.mp4"
cap = cv2.VideoCapture(video_path)


total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

train_end = int(0.7 * total_frames)
val_end = int(0.85 * total_frames)


base_path = "/content/dataset"

splits = ["train", "val", "test"]
for split in splits:
    os.makedirs(f"{base_path}/{split}/fog", exist_ok=True)

count = 0
saved = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break


    if count % 5 != 0:
        count += 1
        continue


    frame = cv2.resize(frame, (256, 256))

    filename = f"frame_{saved:05d}.jpg"

    if count < train_end:
        path = f"{base_path}/train/fog/{filename}"
    elif count < val_end:
        path = f"{base_path}/val/fog/{filename}"
    else:
        path = f"{base_path}/test/fog/{filename}"

    cv2.imwrite(path, frame)

    saved += 1
    count += 1

cap.release()

print(saved, "frames extracted and split")

1126 frames extracted and split


In [29]:
import cv2
import os

video_path = "/content/Highway_5_RGB.mp4"
cap = cv2.VideoCapture(video_path)


total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

train_end = int(0.7 * total_frames)
val_end = int(0.85 * total_frames)


base_path = "/content/dataset"

splits = ["train", "val", "test"]
for split in splits:
    os.makedirs(f"{base_path}/{split}/clear", exist_ok=True)

count = 0
saved = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break


    if count % 5 != 0:
        count += 1
        continue


    frame = cv2.resize(frame, (256, 256))

    filename = f"frame_{saved:05d}.jpg"

    if count < train_end:
        path = f"{base_path}/train/clear/{filename}"
    elif count < val_end:
        path = f"{base_path}/val/clear/{filename}"
    else:
        path = f"{base_path}/test/clear/{filename}"

    cv2.imwrite(path, frame)

    saved += 1
    count += 1

cap.release()

print(saved, "frames extracted and split")

1126 frames extracted and split


In [49]:
from torch.utils.data import Dataset
import os
import cv2

class DehazeDataset(Dataset):
    def __init__(self, fog_dir, clear_dir, transform=None):
        self.fog_dir = fog_dir
        self.clear_dir = clear_dir
        self.images = sorted(os.listdir(fog_dir))
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
      name = self.images[idx]
      fog = cv2.imread(os.path.join(self.fog_dir, name))
      clear = cv2.imread(os.path.join(self.clear_dir, name))


      fog = cv2.cvtColor(fog, cv2.COLOR_BGR2RGB)
      clear = cv2.cvtColor(clear, cv2.COLOR_BGR2RGB)
      fog = torch.from_numpy(fog).permute(2, 0, 1).float() / 255.0
      clear = torch.from_numpy(clear).permute(2, 0, 1).float() / 255.0

      return fog, clear

In [50]:
import os, shutil, random

base = "/content/dataset"

train_fog = f"{base}/train/fog"
train_clear = f"{base}/train/clear"

val_fog = f"{base}/val/fog"
val_clear = f"{base}/val/clear"

test_fog = f"{base}/test/fog"
test_clear = f"{base}/test/clear"

exp_base = f"{base}/experiment"

os.makedirs(f"{exp_base}/train/fog", exist_ok=True)
os.makedirs(f"{exp_base}/train/clear", exist_ok=True)

os.makedirs(f"{exp_base}/val/fog", exist_ok=True)
os.makedirs(f"{exp_base}/val/clear", exist_ok=True)

os.makedirs(f"{exp_base}/test/fog", exist_ok=True)
os.makedirs(f"{exp_base}/test/clear", exist_ok=True)

images = sorted(os.listdir(train_fog))
random.shuffle(images)

train_imgs = images[:100]
val_imgs = images[100:110]
test_imgs = images[110:112]


def copy_files(img_list, src_fog, src_clear, dst_fog, dst_clear):
  for img in img_list:
    shutil.copy(os.path.join(src_fog, img), os.path.join(dst_fog, img))
    shutil.copy(os.path.join(src_clear, img), os.path.join(dst_clear, img))

copy_files(train_imgs, train_fog, train_clear,
        f"{exp_base}/train/fog", f"{exp_base}/train/clear")

copy_files(val_imgs, train_fog, train_clear,
        f"{exp_base}/val/fog", f"{exp_base}/val/clear")

copy_files(test_imgs, train_fog, train_clear,
        f"{exp_base}/test/fog", f"{exp_base}/test/clear")

print("Experiment dataset created")

Experiment dataset created


In [51]:
criterion = nn.L1Loss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-5
)

print("Loss and optimizer ready")

Loss and optimizer ready


In [63]:
from torch.utils.data import DataLoader

train_dataset = DehazeDataset(
    "/content/dataset/train/fog",
    "/content/dataset/train/clear"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

epochs = 5
scaler = torch.cuda.amp.GradScaler()

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for fog, clear in train_loader:
    fog = fog.to(device, non_blocking=True)
    clear = clear.to(device, non_blocking=True)

    optimizer.zero_grad()

    with torch.cuda.amp.autocast():
      output = model(fog)
      loss = criterion(output, clear)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    total_loss += loss.item()

  avg_loss = total_loss / len(train_loader)

  print(f"Epoch {epoch+1}/{epochs}  Loss: {avg_loss:.4f}")

/tmp/ipykernel_195/134302369.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_195/134302369.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


Epoch 1/5  Loss: 0.2328
Epoch 2/5  Loss: 0.2195
Epoch 3/5  Loss: 0.2180
Epoch 4/5  Loss: 0.2170
Epoch 5/5  Loss: 0.2160


In [64]:
save_path = "/content/aod_finetuned.pth"

torch.save(model.state_dict(), save_path)

print("Model saved to:", save_path)

Model saved to: /content/dehazeformer_finetuned.pth


In [67]:
test_dataset = DehazeDataset(
    "/content/dataset/test/fog",
    "/content/dataset/test/clear"
)

test_loader = DataLoader(test_dataset, batch_size=1)

output_dir = "/content/outputs/synthetic"
os.makedirs(output_dir, exist_ok=True)

model.eval()

with torch.no_grad():
    for i, (fog, clear) in enumerate(test_loader):

        fog = fog.to(device)

        output = model(fog)

        save_image(output, f"{output_dir}/output_{i:04d}.png")

print("Synthetic test images saved.")

Synthetic test images saved.


In [69]:
import os
import cv2

video_dir = "/content/real_dataset"
frames_dir = "/content/real_frames"

os.makedirs(frames_dir, exist_ok=True)

frame_skip = 5  # take every 5th frame

for video_name in os.listdir(video_dir):
    if not video_name.endswith(".mp4"):
        continue

    video_path = os.path.join(video_dir, video_name)
    cap = cv2.VideoCapture(video_path)

    count = 0
    saved = 0

    video_id = video_name.split(".")[0]

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if count % frame_skip != 0:
            count += 1
            continue

        frame = cv2.resize(frame, (256, 256))

        filename = f"{video_id}_frame_{saved:05d}.jpg"
        cv2.imwrite(os.path.join(frames_dir, filename), frame)

        saved += 1
        count += 1

    cap.release()
    print(f"{video_name} → {saved} frames extracted")

print("All videos processed ")

Dataset9.mp4 → 147 frames extracted
Dataset3.mp4 → 237 frames extracted
Dataset4.mp4 → 426 frames extracted
Dataset7.mp4 → 363 frames extracted
Dataset5.mp4 → 238 frames extracted
Dataset1.mp4 → 508 frames extracted
Dataset8.mp4 → 124 frames extracted
Dataset10.mp4 → 345 frames extracted
Dataset2.mp4 → 217 frames extracted
Dataset6.mp4 → 123 frames extracted
All videos processed 


In [71]:
import os
import cv2
from torchvision import transforms
import numpy as np

input_dir = "/content/real_frames"
output_dir = "/content/outputs/real"

os.makedirs(output_dir, exist_ok=True)

transform = transforms.ToTensor()

model.eval()

for img_name in os.listdir(input_dir):

    img_path = os.path.join(input_dir, img_name)

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w = img.shape[:2]

    h = h - h % 8
    w = w - w % 8
    img = img[:h, :w]

    tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(tensor)

    output = output.clamp(0,1)
    output = output.squeeze().permute(1,2,0).cpu().numpy()
    output = (output*255).astype(np.uint8)

    output = cv2.cvtColor(output, cv2.COLOR_RGB2BGR)

    cv2.imwrite(os.path.join(output_dir, img_name), output)

print("Real video frames dehazed.")

Real video frames dehazed.


In [73]:
gt_files = sorted(os.listdir(gt_dir))
pred_files = sorted(os.listdir(pred_dir))

psnr_list = []
ssim_list = []

for gt_name, pred_name in zip(gt_files, pred_files):

    gt_path = os.path.join(gt_dir, gt_name)
    pred_path = os.path.join(pred_dir, pred_name)

    gt = cv2.imread(gt_path)
    pred = cv2.imread(pred_path)

    gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]))

    gt = cv2.cvtColor(gt, cv2.COLOR_BGR2RGB)
    pred = cv2.cvtColor(pred, cv2.COLOR_BGR2RGB)

    gt = gt.astype(np.float32) / 255.0
    pred = pred.astype(np.float32) / 255.0

    psnr = peak_signal_noise_ratio(gt, pred, data_range=1.0)
    ssim = structural_similarity(gt, pred, channel_axis=2, data_range=1.0)

    psnr_list.append(psnr)
    ssim_list.append(ssim)

print("Average PSNR:", np.mean(psnr_list))
print("Average SSIM:", np.mean(ssim_list))

Average PSNR: 11.708135217804664
Average SSIM: 0.71754545


In [74]:
# real dataset

input_dir = "/content/real_frames"
output_dir = "/content/outputs/real"

sharp_input = []
sharp_output = []

for name in os.listdir(input_dir):
  in_path = os.path.join(input_dir, name)
  out_path = os.path.join(output_dir, name)

  if not os.path.exists(out_path):
      continue

  inp = cv2.imread(in_path, 0)
  out = cv2.imread(out_path, 0)

  s1 = cv2.Laplacian(inp, cv2.CV_64F).var()
  s2 = cv2.Laplacian(out, cv2.CV_64F).var()

  sharp_input.append(s1)
  sharp_output.append(s2)

print("Average sharpness input:", np.mean(sharp_input))
print("Average sharpness output:", np.mean(sharp_output))

Average sharpness input: 362.6860465173538
Average sharpness output: 641.623915546918


In [75]:
import shutil

shutil.make_archive("/content/outputs", 'zip', "/content/outputs")

'/content/outputs.zip'

In [76]:
from google.colab import files

files.download("/content/outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>